# SHA-star + OBP

Notebook ini mencoba mencari jalur dari start ke goal dengan menghindari obstacle.

Urutannya: buat peta → cari OBP → A* 16 arah → gabungkan jalur → tampilkan hasil.

Catatan: bentuk peta dan beberapa detail OBP adalah pendekatan implementasi, bukan salinan persis simulasi jurnal.

In [ ]:
import math
import heapq
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch

# Parameter peta dan robot
MAP_MIN = 0
MAP_MAX = 200
ROBOT_RADIUS = 2
R1 = 1.5 * ROBOT_RADIUS
CLEARANCE = ROBOT_RADIUS
Q = 2
C = 1
OBP_OFFSET = R1 + 1

In [ ]:
# Dua peta obstacle yang dipakai untuk percobaan
MAPS = {
    1: {
        "name": "Sharp",
        "obstacles": [
            [(0, 40), (120, 80), (0, 80)],
            [(50, 140), (200, 100), (200, 140)]
        ]
    },
    2: {
        "name": "Narrow",
        "obstacles": [
            [(0, 40), (120, 40), (120, 60), (0, 60)],
            [(0, 85), (90, 85), (90, 105), (0, 105)],
            [(100, 85), (200, 85), (200, 105), (100, 105)],
            [(50, 130), (200, 130), (200, 150), (50, 150)]
        ]
    }
}

def jarak(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def titik_dalam_poligon(titik, poligon):
    x, y = titik
    dalam = False
    for i in range(len(poligon)):
        x1, y1 = poligon[i]
        x2, y2 = poligon[(i + 1) % len(poligon)]
        if (y1 > y) != (y2 > y):
            x_potong = x1 + (y - y1) * (x2 - x1) / (y2 - y1)
            if x < x_potong:
                dalam = not dalam
    return dalam

def jarak_titik_ke_ruas(p, a, b):
    dx, dy = b[0] - a[0], b[1] - a[1]
    panjang2 = dx * dx + dy * dy
    if panjang2 == 0:
        return jarak(p, a)
    t = ((p[0] - a[0]) * dx + (p[1] - a[1]) * dy) / panjang2
    t = max(0, min(1, t))
    titik_dekat = (a[0] + t * dx, a[1] + t * dy)
    return jarak(p, titik_dekat)

def jarak_ke_obstacle(titik, poligon):
    if titik_dalam_poligon(titik, poligon):
        return 0
    return min(jarak_titik_ke_ruas(titik, poligon[i], poligon[(i+1) % len(poligon)])
               for i in range(len(poligon)))

def titik_aman(titik, obstacles):
    x, y = titik
    if not (MAP_MIN <= x <= MAP_MAX and MAP_MIN <= y <= MAP_MAX):
        return False
    return all(jarak_ke_obstacle(titik, p) > CLEARANCE for p in obstacles)

def garis_aman(a, b, obstacles):
    # Cek beberapa titik di sepanjang garis.
    jumlah = max(2, int(jarak(a, b) * 2))
    for t in np.linspace(0, 1, jumlah):
        titik = (a[0] + (b[0] - a[0]) * t,
                 a[1] + (b[1] - a[1]) * t)
        if not titik_aman(titik, obstacles):
            return False
    return True

## 1. Mencari OBP

Program mengambil beberapa kandidat di sekitar sudut obstacle. Kandidat yang terlalu dekat obstacle dibuang. Setelah itu, dipilih kandidat yang dekat dengan titik potong dan bisa menyambung lurus ke goal dengan aman.

Untuk menjaga kode tetap mudah diikuti, kandidat diambil dari sudut poligon yang digeser ke luar. Ini pendekatan sederhana untuk mencari OBP.

In [ ]:
def cari_kandidat_obp(obstacles):
    kandidat = []
    for poligon in obstacles:
        tengah_x = sum(p[0] for p in poligon) / len(poligon)
        tengah_y = sum(p[1] for p in poligon) / len(poligon)

        # Ambil titik di sepanjang sisi obstacle, bukan hanya sudutnya.
        for i in range(len(poligon)):
            a = poligon[i]
            b = poligon[(i + 1) % len(poligon)]
            panjang_sisi = jarak(a, b)
            jumlah = max(1, int(panjang_sisi / 10))

            for k in range(jumlah + 1):
                t = k / jumlah
                x = a[0] + (b[0] - a[0]) * t
                y = a[1] + (b[1] - a[1]) * t
                dx, dy = x - tengah_x, y - tengah_y
                panjang = math.hypot(dx, dy)
                if panjang == 0:
                    continue
                titik = (x + OBP_OFFSET * dx / panjang,
                         y + OBP_OFFSET * dy / panjang)
                if titik_aman(titik, obstacles):
                    kandidat.append(titik)

    return kandidat

def cari_potongan(awal, tujuan, obstacles):
    jumlah = max(2, int(jarak(awal, tujuan) * 2))
    sebelumnya = awal
    for t in np.linspace(0, 1, jumlah):
        titik = (awal[0] + (tujuan[0] - awal[0]) * t,
                 awal[1] + (tujuan[1] - awal[1]) * t)
        if not all(jarak_ke_obstacle(titik, p) > 0 for p in obstacles):
            return sebelumnya
        sebelumnya = titik
    return tujuan

def cari_obp(awal, tujuan, obstacles):
    if garis_aman(awal, tujuan, obstacles):
        return awal, []

    kandidat = cari_kandidat_obp(obstacles)
    if not kandidat:
        raise ValueError("Kandidat OBP tidak ditemukan.")

    potongan = cari_potongan(awal, tujuan, obstacles)
    # Pilih kandidat yang dekat dengan potongan dan punya garis aman ke goal.
    bisa_dipakai = []
    for titik in kandidat:
        if garis_aman(titik, tujuan, obstacles):
            nilai = jarak(titik, potongan)
            bisa_dipakai.append((nilai, titik))

    if not bisa_dipakai:
        raise ValueError("Tidak ada kandidat OBP yang bisa tersambung langsung ke goal.")

    bisa_dipakai.sort(key=lambda item: item[0])
    return bisa_dipakai[0][1], kandidat

## 2. Danger coefficient dan A* 16 arah

A* memilih node dengan nilai `f = g + h + danger`. `g` adalah biaya dari start, `h` perkiraan jarak ke OBP, dan `danger` memberi tambahan biaya jika node dekat obstacle.

In [ ]:
def danger(titik, obstacles):
    nilai = 0
    for poligon in obstacles:
        d = jarak_ke_obstacle(titik, poligon)
        if 0 < d <= R1:
            nilai += (Q / d) ** C
    return nilai

GERAK = [
    (1, 0), (-1, 0), (0, 1), (0, -1),
    (1, 1), (1, -1), (-1, 1), (-1, -1),
    (2, 1), (2, -1), (-2, 1), (-2, -1),
    (1, 2), (1, -2), (-1, 2), (-1, -2)
]

def astar(awal, obp, obstacles):
    awal = (int(awal[0]), int(awal[1]))
    tujuan = (int(round(obp[0])), int(round(obp[1])))
    antrean = [(0, awal)]
    biaya = {awal: 0}
    induk = {}
    sudah = set()

    while antrean:
        _, sekarang = heapq.heappop(antrean)
        if sekarang in sudah:
            continue
        sudah.add(sekarang)

        if jarak(sekarang, obp) < 1.5:
            jalur = [sekarang]
            while sekarang in induk:
                sekarang = induk[sekarang]
                jalur.append(sekarang)
            jalur.reverse()
            if jarak(jalur[-1], obp) > 0.1:
                jalur.append(obp)
            return jalur

        for dx, dy in GERAK:
            baru = (sekarang[0] + dx, sekarang[1] + dy)
            if not (MAP_MIN <= baru[0] <= MAP_MAX and MAP_MIN <= baru[1] <= MAP_MAX):
                continue
            if not titik_aman(baru, obstacles) or not garis_aman(sekarang, baru, obstacles):
                continue

            biaya_baru = biaya[sekarang] + jarak(sekarang, baru) + danger(baru, obstacles)
            if biaya_baru < biaya.get(baru, float("inf")):
                biaya[baru] = biaya_baru
                induk[baru] = sekarang
                h = jarak(baru, obp)
                heapq.heappush(antrean, (biaya_baru + h, baru))

    raise ValueError("A* tidak menemukan jalur ke OBP.")

## 3. Menggabungkan jalur dan menggambar hasil

In [ ]:
def rapikan_jalur(jalur, obstacles):
    if len(jalur) <= 2:
        return jalur
    hasil = [jalur[0]]
    i = 0
    while i < len(jalur) - 1:
        j = len(jalur) - 1
        while j > i + 1 and not garis_aman(jalur[i], jalur[j], obstacles):
            j -= 1
        hasil.append(jalur[j])
        i = j
    return hasil

def gambar_hasil(map_id, awal, tujuan, obp, jalur, obstacles):
    fig, ax = plt.subplots(figsize=(8, 6))
    for poligon in obstacles:
        ax.add_patch(PolygonPatch(poligon, closed=True, color="black"))

    ax.plot([p[0] for p in jalur], [p[1] for p in jalur],
            color="red", marker=".", label="Jalur")
    ax.scatter(*awal, color="limegreen", label="Start")
    ax.scatter(*tujuan, color="blue", label="Goal")
    ax.scatter(*obp, color="orange", marker="*", s=150, label="OBP akhir")

    ax.set_title("Hasil SHA-star + OBP - " + MAPS[map_id]["name"])
    ax.set_xlim(0, 200)
    ax.set_ylim(200, 0)
    ax.set_aspect("equal")
    ax.grid(alpha=0.2)
    ax.legend()
    plt.show()

def jalankan(map_id=1, awal=(10, 10), tujuan=(190, 190)):
    obstacles = MAPS[map_id]["obstacles"]

    if not titik_aman(awal, obstacles):
        raise ValueError("Start berada di obstacle atau terlalu dekat.")
    if not titik_aman(tujuan, obstacles):
        raise ValueError("Goal berada di obstacle atau terlalu dekat.")

    obp, kandidat = cari_obp(awal, tujuan, obstacles)
    if obp == awal:
        jalur = [awal, tujuan]
    else:
        jalur_lokal = astar(awal, obp, obstacles)
        if not garis_aman(obp, tujuan, obstacles):
            raise ValueError("OBP belum bisa tersambung lurus ke goal.")
        jalur = jalur_lokal + [tujuan]

    jalur = rapikan_jalur(jalur, obstacles)
    print("Map:", MAPS[map_id]["name"])
    print("Jumlah kandidat OBP:", len(kandidat))
    print("OBP akhir:", obp)
    print("Panjang jalur:", round(sum(jarak(jalur[i], jalur[i+1]) for i in range(len(jalur)-1)), 2))
    print("Jalur bebas obstacle:", all(garis_aman(jalur[i], jalur[i+1], obstacles) for i in range(len(jalur)-1)))
    gambar_hasil(map_id, awal, tujuan, obp, jalur, obstacles)
    return {"obp": obp, "jalur": jalur}

# Ganti map_id menjadi 2 untuk mencoba peta narrow
hasil = jalankan(map_id=1, awal=(10, 10), tujuan=(190, 190))